# Orfeo Toolbox: BundleToPerfectSensor Pansharpening

This workbook provides code to pansharpen very high-resolution optical satellite imagery. The code provides options to perform pansharpening for a single image pair (multispectral and panchromatic image) and to loop through a number of images pairs to pansharpen multiple images consecutively.

A satellite image is made up of lots of pixels, imagine a grid like structure, containing squares of information. Each pixel (square) is the equivalent to a measurable distance on the ground, this is called spatial resolution. For example, if an image is defined as having 0.3 m resolution, then each pixel in the image is equivalent to 0.3 m ground sampling distance.

Panchromatic images are grayscale images captured across the visible (and possibly near-infrared) wavelengths of the electromagnetic spectrum. Multispectral images cover more bands of the electromagnetic spectrum but each band covers a narrow portion of the visible, near-infrared, and shortwave infrared wavelengths of the spectrum. A panchromatic image will have a higher resolution than a multispectral image as it captures a wider range of light in a single band, allowing it to be significantly sharper. For example, Vantor's WorldView 3 sensor, produces a panchromatic image with a spatial resolution of 0.3 m, and a multispectral image with a spatial resolution of 1.24 m. 

To analyse a multispectral image, it is useful to enhance its spatial resolution to the higher spatial resolution of the panchromatic image, through the process of pansharpening. Pansharpening takes the panchromatic and multispectral images, and where the two rasters fully overlap, fuses them together to produce a multispectral image with the higher spatial resolution of the panchromatic image (Bovolo et al., 2010). Pansharpening is useful to studying cetacean strandings from space as both spectral and spatial richness is needed to discriminate individual carcass with confidence. Here we use a plugin called OTB (BundleToPerfectSensor) to perform pansharpening.

**Python tips**

For anyone new to Python / coding, here are some useful tips to navigate Jupyter Notebooks:
- when a cell is blue, it is command mode (binding the keys to notebook level commands, when in command mode press h to view all the notebook level commands), when a cell is green, it is in edit mode (allowing you to type code and text in a cell)
- to run a code cell, double click a cell (or whenthe cell is blue press the enter key) to enter edit mode, press the shift + enter keys
- to create a new cell, (in command mode) a inserts a new cell above and b inserts a new cell below
- to save the notebook (if you are in edit mode, press the esc key to exit, the cell will turn blue) and press the s key
- jupyter where possible will auto-complete your code, begin typing and press tab
- hash # indicates comments to help you with code

## **IMPORTANT: DATA FORMAT**

The code requires .tif image format.

Vantor (formerly Maxar Technologies) may provide large image files as multiple .tif files, these are usually accompanied with a single .til file compiling all .tif files. 

The .til file does not contain the image itself, rather a reference to the multiple .tif files. Therefore, loading a .til file directly will not work with this code. 

This code expects a user to have a single .tif multispectral and panchromatic image pair. If a user has multiple .tif files please use the code in the strandings_from_space pipeline to mosaic the multiple .tif files into a single image.

To loop through multiple pairs of multispectral and panchromatic images, the code expects the names to have the same name with exception to including MUL to indicate multispectral and PAN to indicate panchromatic. For example, mosaic_050010168010_01_P001_MUL and mosaic_050010168010_01_P001_PAN. The code can then identify and match pairs to process. The user can amend the indicator the code should look for to find match pairs or if required manually amend the filenames to include MUL or PAN.

## Import packages

In [ ]:
import os
from pathlib import Path
import otbApplication as otb

## Set working directories

In [ ]:
# set working directory to your desktop
# the code automatically selects your desktop
# for another location please amend the file path for main_path below
# get user's home directory
home_directory = os.path.expanduser('~')

# append the desktop folder to the home directory
desktop_directory = os.path.join(home_directory, 'Desktop')

# set and store the desktop as the working directory
os.chdir(desktop_directory)
print(f"The current working directory is set to: {os.getcwd()}")
main_path = os.getcwd()

# confirm whether the main_path exists
if os.path.exists(main_path):
    print(main_path, 'is a correct path'),
else:
    print(main_path, 'is not a correct path')

In [ ]:
# create a number of folders to host the inputs and outputs of the code run through this workbook
folders = [
    'strandings_from_space/inputs',
    'strandings_from_space/outputs',
    'strandings_from_space/temp_outputs'
]
# check if the directory folder exists or not
# if the directory is not present, then create it
for folder in folders:
    os.makedirs(os.path.join(main_path, folder), exist_ok=True)

In [ ]:
# set the working path directories required throughout
input_path = os.path.join(main_path,'strandings_from_space\\inputs')
output_path = os.path.join(main_path,'strandings_from_space\\outputs')
temp_output_path = os.path.join(main_path,'strandings_from_spaces\\temp_outputs')

## Pansharpening a single image pair (multispectral and panchromatic image)

**To run this section of code, ensure only a single image pair are in the 'strandings_from_space\\inputs' folder.** <br>

Load an image pair into the 'strandings_from_space\\inputs' folder now, before continuing to process the following code. <br>

**To run the following code, ensure to manually amend the file names for 'inp', 'inxs' and 'out'.**

In [ ]:
# define the pansharpening method you wish to apply to your image

app = otb.Registry.CreateApplication('BundleToPerfectSensor')

# define the panchromatic image 'inp'
# define the multispectral image 'inxs
# define the output 'out' panshaprned image file name

# ***AMEND LINKS FOR 'inp', 'inxs' and 'out'***

pan_path = os.path.join(input_path, "insert_panchromatic_filename.tif")
mul_path = os.path.join(input_path, "insert_multispectral_filename.tif")
pxs_path = os.path.join(output_path, "insert_pansharpen_filename.tif")

app.SetParameterString('inp', pan_path)
app.SetParameterString('inxs', mul_path)
app.SetParameterString('out', pxs_path)

# set the output image pixel type

app.SetParameterOutputImagePixelType('out', 1)

# execute the pansharpening algorithm and create the pansharpened image file

app.ExecuteAndWriteOutput()

## Pansharpening multiple images pairs (multispectral and panchromatic image) consecutively (loop)

**To run this section of code, multiple image pairs can be placed in the 'strandings_from_space\\inputs' folder.** <br>

Load all the image pairs into the 'strandings_from_space\\inputs' folder now, before continuing to process the following code.

In [ ]:
# collect all .tif files in the 'strandings_from_space\\inputs' folder
tif_files = list(Path(input_path).glob("*.tif"))
print(tif_files)

In [ ]:
# separate MUL and PAN files by those with MUL in their name and those with PAN in their name
# this code expects filenames to differ only by MUL and PAN 
# e.g. mosaic_050010168010_01_P001_MUL and mosaic_050010168010_01_P001_PAN
mul_files = {f.stem.replace("_MUL", ""): f for f in tif_files if "_MUL" in f.stem}
pan_files = {f.stem.replace("_PAN", ""): f for f in tif_files if "_PAN" in f.stem}
print(f'MUL files: {mul_files}')
print(f'PAN files: {pan_files}')

In [ ]:
# create a definition to pansharpen imagery using Orfeo Toolbox BundleToPerfectSensor
def pansharpening (mul_file, pan_file, output_path):

    # define the pansharpening method you wish to apply to your image

    app = otb.Registry.CreateApplication('BundleToPerfectSensor')

    pxs_rename = pan_file.name[:-7] + 'pansharpened22052025.tif'
    output_pxs = os.path.join(output_path, pxs_rename)

    # define the panchromatic image 'inp'
    # define the multispectral image 'inxs
    # define the output 'out' panshaprned image file name

    app.SetParameterString('inp', str(pan_file))
    app.SetParameterString('inxs', str(mul_file))
    app.SetParameterString('out', str(output_pxs))

    # set the output image pixel type

    app.SetParameterOutputImagePixelType('out', 1)

    # execute the pansharpening algorithm and create the pansharpened image file

    app.ExecuteAndWriteOutput()

In [ ]:
# loop through matching multispectral and panchromatic image pairs and perform BundleToPerfectSensor pansharpening
for base_name in mul_files.keys() & pan_files.keys():
    mul_path = mul_files[base_name]
    pan_path = pan_files[base_name]
    
    print(f"Processing pair: {mul_path.name} and {pan_path.name}")
    pansharpening(mul_path, pan_path, output_path)

## Using pyotb to pansharpen

import otbApplication as otb is the official Orfeo Toolbox Python API, whereas import pyotb is a higher-level interface that makes using Orfeo Toolbox in Python more intuitive.

The following code provides an example of how the same pansharpening algorithm, BundleToPerfectSensor, can be executed using pyotb, for a single multispectral and panchromatic image pair.

In [ ]:
# run on first use only to install Orfeo Toolbox, thereafter hash code or remove cell from workbook
!pip install pyotb --upgrade

In [ ]:
# import the required packages
import os
# set the environmental variable as the location of Orfeo Toolbox in your operating system
# ***AMEND LINK***
os.environ['OTB_ROOT'] = 'C:\\Users\\Desktop\\strandings_from_space\\OTB-8.0.1-Win64'
import pyotb
from typing import Any, Dict, List, Optional

***Run the cells above to set the working directories now, if not already done so.***

In [ ]:
# import the panchromatic and multispectral imagery to pansharpen
output_pan = os.path.join(input_path, 'insert_panchromatic_filename.tif')
output_mul = os.path.join(input_path, 'insert_multispectral_filename.tif')
input_pan = pyotb.Input(output_pan)
input_mul = pyotb.Input(output_mul)

In [ ]:
# perform the bundle to perfect raster pansharpening algorithm
pxs = pyotb.BundleToPerfectSensor(inp=input_pan, inxs=input_mul, method='bayes', mode="default")

In [ ]:
# take the input pan file name, remove 'PAN.tif' and replacing with 'pansharpened.tif'
# join with main_output link to create an output path
dir_pan,file_pan = os.path.split(output_pan)
pxs_rename = file_pan[:-7] + 'pansharpened.tif'
output_pxs = os.path.join(output_path, pxs_rename)

In [ ]:
# write the pansharpened image to the main output folder
pxs.write(output_pxs)